In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from PIL import Image

In [ ]:
def cargar_imagen(ruta, size=(500, 500)):
    img = Image.open(ruta)
    # Convertir a RGB si la imagen tiene canal alfa u otro modo
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img_original = img.copy()  # Copia explícita de la imagen original
    img_redimensionada = img.resize(size)
    return img_original, img_redimensionada

In [ ]:
def muestrear_pixeles(img_array, n_muestra=500000):
    pixels = img_array.reshape(-1, 3)
    # Asegurar que n_muestra no exceda el número de píxeles disponibles
    n_muestra = min(n_muestra, pixels.shape[0])
    idx = np.random.choice(pixels.shape[0], n_muestra, replace=False)
    return pixels[idx]

In [ ]:
def obtener_k_optimo(pixels, k_min=3, k_max=10):
    mejor_k = k_min
    mejor_score = -1
    for k in range(k_min, k_max+1):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(pixels)
        score = silhouette_score(pixels, labels, sample_size=1000, random_state=42)
        if score > mejor_score:
            mejor_score = score
            mejor_k = k
    return mejor_k

In [ ]:
def rgb_to_hex(rgb):
    """Convierte RGB a código hexadecimal"""
    return '#{:02X}{:02X}{:02X}'.format(rgb[0], rgb[1], rgb[2])

def get_text_color(rgb):
    luminancia = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
    return 'white' if luminancia < 140 else 'black'

def calcular_porcentajes(labels, k):
    """Calcula el porcentaje de cada color en la imagen"""
    unique, counts = np.unique(labels, return_counts=True)
    total = len(labels)
    porcentajes = {i: 0 for i in range(k)}
    for u, c in zip(unique, counts):
        porcentajes[u] = (c / total) * 100
    return porcentajes

In [ ]:
def aplicar_kmeans(img_array, k):
    """Aplica K-Means a los píxeles de la imagen y retorna los colores dominantes"""
    pixels = img_array.reshape(-1, 3)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(pixels)
    colores = np.clip(kmeans.cluster_centers_, 0, 255).astype(np.uint8)
    return colores, labels

def crear_imagen_segmentada(colores, labels, shape):
    """Crea la imagen segmentada usando los colores del clustering"""
    return colores[labels].reshape(shape).astype(np.uint8)

def ordenar_colores_por_porcentaje(colores, porcentajes, k):
    """Ordena los colores de mayor a menor según su porcentaje en la imagen"""
    indices_ordenados = sorted(range(k), key=lambda i: porcentajes[i], reverse=True)
    colores_ordenados = colores[indices_ordenados]
    porcentajes_ordenados = [porcentajes[i] for i in indices_ordenados]
    return colores_ordenados, porcentajes_ordenados

In [59]:
def dibujar_cuadros_colores(fig, gs, colores, porcentajes):
    """Dibuja los cuadros de colores individuales en la fila superior"""
    from matplotlib.patches import FancyBboxPatch

    for i, (color, pct) in enumerate(zip(colores, porcentajes)):
        ax = fig.add_subplot(gs[0, i])

        # Normalizar color a rango [0, 1] para matplotlib
        color_normalizado = np.clip(color / 255.0, 0, 1)
        rect = FancyBboxPatch((0.05, 0.1), 0.9, 0.8,
                               boxstyle="round,pad=0.02,rounding_size=0.15",
                               facecolor=color_normalizado,
                               edgecolor='#333333', linewidth=2)
        ax.add_patch(rect)

        # Texto del código HEX dentro del cuadro
        hex_color = rgb_to_hex(color)
        text_color = get_text_color(color)
        ax.text(0.5, 0.5, hex_color, fontsize=11, fontweight='bold',
                ha='center', va='center', color=text_color,
                transform=ax.transAxes)

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        ax.set_facecolor('white')

In [61]:
def dibujar_barra_proporcion(fig, gs, colores, porcentajes):
    """Dibuja la barra de proporción horizontal con los colores"""
    ax_barra = fig.add_subplot(gs[1, :])
    ax_barra.set_facecolor('white')

    # Crear barra de proporción horizontal
    x_start = 0
    for color, pct in zip(colores, porcentajes):
        width = pct / 100
        color_normalizado = np.clip(color / 255.0, 0, 1)
        rect = plt.Rectangle((x_start, 0.3), width, 0.4,
                              facecolor=color_normalizado,
                              edgecolor='#333333', linewidth=1)
        ax_barra.add_patch(rect)

        # Etiqueta de porcentaje (solo si es mayor al 5%)
        if pct > 5:
            text_color = get_text_color(color)
            ax_barra.text(x_start + width/2, 0.5, f'{pct:.1f}%',
                         fontsize=12, fontweight='bold', ha='center', va='center',
                         color=text_color)
        x_start += width

    # Información RGB debajo de la barra
    x_start = 0
    for color, pct in zip(colores, porcentajes):
        width = pct / 100
        rgb_text = f'RGB({color[0]}, {color[1]}, {color[2]})'
        ax_barra.text(x_start + width/2, 0.1, rgb_text,
                     fontsize=9, ha='center', va='center', color='#555555')
        x_start += width

    ax_barra.set_xlim(0, 1)
    ax_barra.set_ylim(0, 0.8)
    ax_barra.set_title('Distribución de Colores en la Imagen', fontsize=16,
                       color='#333333', pad=15, fontweight='bold')
    ax_barra.axis('off')


In [ ]:
def dibujar_imagenes(fig, gs, img_orig, img_redim, img_segmentada, k):
    """Dibuja las tres imágenes: original, redimensionada y segmentada"""
    titulos = ['Imagen Original', 'Imagen Redimensionada', 'Imagen Segmentada']
    imagenes = [img_orig, img_redim, img_segmentada]

    for idx, (titulo, img) in enumerate(zip(titulos, imagenes)):
        if k >= 3:
            start_col = idx * (k // 3)
            end_col = start_col + (k // 3)
            if idx == 2:  # Última imagen toma las columnas restantes
                end_col = k
            ax = fig.add_subplot(gs[2, start_col:end_col])
        else:
            ax = fig.add_subplot(gs[2, :])

        ax.imshow(img)
        ax.set_title(titulo, fontsize=14, color='#333333', pad=10, fontweight='bold')

        # Estilo del marco
        for spine in ax.spines.values():
            spine.set_edgecolor('#333333')
            spine.set_linewidth(1.5)
        ax.tick_params(colors='#333333', length=0)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor('white')

In [63]:
def crear_figura_paleta(k):
    """Crea y configura la figura principal para la visualización"""
    plt.style.use('default')
    fig = plt.figure(figsize=(18, 14), facecolor='white')

    gs = fig.add_gridspec(3, k, height_ratios=[0.8, 2.5, 2], hspace=0.4, wspace=0.15,
                          left=0.05, right=0.95, top=0.92, bottom=0.05)

    fig.suptitle('Paleta de Colores Extraída', fontsize=24, fontweight='bold',
                 color='#333333', y=0.98)

    return fig, gs

In [64]:
resultado = crear_figura_paleta('image1.jpg')
print("\nResumen de la paleta:")
for i, (hex_c, pct) in enumerate(zip(resultado['colores_hex'], resultado['porcentajes']), 1):
    print(f"   Color {i}: {hex_c} - {pct:.1f}%")

ValueError: Number of columns must be a positive integer, not 'image1.jpg'

<Figure size 1800x1400 with 0 Axes>

In [ ]:

# 1. Cargar imagen y mostrar original/preprocesada



# 2. Extraer píxeles y muestrear aleatoriamente


# 3. Determinar k óptimo con silueta


# 4. Funciones auxiliares para visualización profesional



# 5. Aplicar K-Means y mostrar resultados profesionales



# Ejemplo de uso:

